# ASTRA QuickStart

Generate a **prebiotic synthesis network** for a target molecule, guided by *deep research* you
collect yourself from a chat assistant.

**The flow:**
1. Pick a **target molecule** (Cell: *Step 1*).
2. Copy the **deep-research prompt** this notebook prints and run it in one of
   **Claude.ai / Gemini / ChatGPT** (their *Deep Research* / *Research* mode) — *Step 2*.
3. **Paste** the markdown answer back in — *Step 3*.
4. **Run** the ASTRA generation pipeline — *Step 4*.
5. See the resulting **synthesis network** — *Step 5*.

> **Prerequisites**
> - Dependencies installed: `pip install -r requirements.txt`
> - `config.env` holds a valid API key for the model backend you choose below
>   (for `claude` that's `ANTHROPIC_API_KEY`). This repo's `config.env` is already populated.
>
> This notebook only runs **generation** — there is no validation step.


In [ ]:
# --- Step 0: environment setup (run this first) ---
import os, sys
from pathlib import Path

# Point at the ASTRA repo root (the folder that holds pipeline.py + config.env).
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pipeline.py").exists():
    PROJECT_ROOT = Path("/Users/daniel/Desktop/astra")   # fallback if launched elsewhere
os.chdir(PROJECT_ROOT)                 # config.env / outputs/ / logs/ resolve relative to cwd
sys.path.insert(0, str(PROJECT_ROOT))

# Choose the LLM backend that BUILDS the network.
#   "claude" | "gpt41" | "gpt5.4_openai" | "gpt5.4_agent_azure"
PROVIDER = "claude"                     # <-- edit me
os.environ["MODEL_PROVIDER"] = PROVIDER  # must be set BEFORE importing the pipeline

from dotenv import load_dotenv
load_dotenv("config.env")               # override=False -> the PROVIDER set above wins

print("Repo:    ", PROJECT_ROOT)
print("Provider:", os.environ["MODEL_PROVIDER"])


## Step 1 — Choose your target molecule

Any common/IUPAC molecule name works (e.g. `Glycine`, `L-Alanine`, `Serine`, `Arginine`,
`Threonine`, `Valine`, `Histidine`). It's used as plain text — not SMILES or InChI.


In [ ]:
TARGET_MOLECULE = "Glycine"     # <-- edit me

# Generation knobs (defaults mirror the pipeline; lower them for faster / cheaper runs)
NUM_PATHWAYS         = 10
CONFIDENCE_THRESHOLD = 0.8
MAX_RETRIES          = 3


## Step 2 — Collect the deep research

Run the prompt printed by the next cell in **one** of these (use their *Deep Research* mode):

| Assistant | Where |
|-----------|-------|
| **Claude.ai** | *Research* |
| **Gemini** | *Deep Research* |
| **ChatGPT** | *Deep Research* |

When it finishes, **copy the entire markdown answer** — you'll paste it in Step 3.


In [ ]:
# --- Print the ready-to-copy deep-research prompt for TARGET_MOLECULE ---
from config.prompts import DEEP_RESEARCH_TARGET_MOLECULE_PROMPT


def build_deep_research_prompt(molecule: str) -> str:
    query = f"""Conduct comprehensive research on the prebiotic synthesis of {molecule}.

Please investigate and provide detailed information on:

1. KNOWN PREBIOTIC SYNTHESIS ROUTES for {molecule}:
   - What pathways have been proposed or demonstrated in the literature?
   - Which routes start from simple molecules (H2O, CO2, NH3, HCN, H2CO, H2S, CH4, H2)?
   - What are the key reaction steps in each route?

2. KEY INTERMEDIATES on the path to {molecule}:
   - What are the essential precursor molecules?
   - Which intermediates serve as branch points for multiple pathways?

3. ENVIRONMENTAL CONDITIONS for synthesis:
   - Hydrothermal vent conditions (temperature, pressure, mineral catalysts)
   - Surface/evaporitic pool conditions (wet-dry cycles, UV, mineral surfaces)
   - Which environment is most favorable for each step?

4. MINERAL CATALYSTS relevant to {molecule} synthesis:
   - Iron-sulfur minerals (greigite, pyrite, magnetite)
   - Clay minerals (montmorillonite)
   - Other relevant catalysts

5. EXPERIMENTAL EVIDENCE:
   - Key laboratory experiments demonstrating relevant reactions
   - Yields, selectivities, and conditions from published studies
   - Miller-Urey type experiments, hydrothermal vent simulations

6. OPEN QUESTIONS AND CHALLENGES:
   - What steps remain unresolved or debated?
   - What are the main bottlenecks in prebiotic synthesis of {molecule}?

Focus on astrochemistry, prebiotic chemistry, and origin of life literature.
Provide specific citations with author names, years, and journal names."""
    return DEEP_RESEARCH_TARGET_MOLECULE_PROMPT.strip() + "\n\n---\n\n" + query


prompt = build_deep_research_prompt(TARGET_MOLECULE)

# Save a copy for easy access, then print it for copying.
out_dir = Path("reports/quickstart")
out_dir.mkdir(parents=True, exist_ok=True)
prompt_path = out_dir / f"{TARGET_MOLECULE}_deep_research_prompt.txt"
prompt_path.write_text(prompt)

print(prompt)
print(f"\n[Prompt also saved to {prompt_path}]")


## Step 3 — Paste the deep-research markdown

Replace the placeholder below with the full markdown you copied in Step 2, then run the two cells.


In [ ]:
DEEP_RESEARCH_CONTENT = r"""
PASTE THE MARKDOWN DEEP-RESEARCH OUTPUT HERE
"""


In [ ]:
# --- Save the pasted research to a cache file the pipeline will read ---
content = DEEP_RESEARCH_CONTENT.strip()
if not content or content.startswith("PASTE THE MARKDOWN"):
    raise ValueError(
        "Paste your deep-research markdown into DEEP_RESEARCH_CONTENT (the cell above), then re-run."
    )

DR_FILE = Path("reports/quickstart") / f"{TARGET_MOLECULE}-Run1.txt"
DR_FILE.parent.mkdir(parents=True, exist_ok=True)
DR_FILE.write_text(content)
print(f"Saved {len(content):,} chars of deep research -> {DR_FILE}")


## Step 4 — Run the ASTRA pipeline

This makes real LLM calls (SynthesisNetworkAgent proposes a network → CriticAgent scores it, up to
`MAX_RETRIES` cycles), so it can take a few minutes. Your pasted research is loaded from disk — no
deep-research API call is made.


In [ ]:
# Jupyter supports top-level `await`. (In a plain .py script you'd use asyncio.run(...).)
from core.network_workflow import run_network_workflow

result, output_path = await run_network_workflow(
    target_molecule=TARGET_MOLECULE,
    enable_deep_research=True,
    deep_research_cache_file=str(DR_FILE),   # <- our pasted markdown; no live DR API call
    num_pathways=NUM_PATHWAYS,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    max_retries=MAX_RETRIES,
    enable_io_logging=True,
    config_label="quickstart",
    run_number=1,
    output_dir="outputs/quickstart",
)
print("Saved network JSON ->", output_path)


## Step 5 — Results


In [ ]:
from pipeline import print_network_results   # safe import (pipeline runs main() only under __main__)

print_network_results(result, output_path)

ev = result.evaluation
print(f"\nConfidence: {ev.overall_confidence:.3f} | Approved: {ev.approved} | Attempts: {result.attempts}")
print(f"Network JSON: {output_path}")
